# Day 10: Train the Baseline 3D-CNN 🛰️

## 1. Objective and Engineering Context
On **Day 8**, we implemented the `Hyperspectral3DCNN` baseline model architecture in PyTorch. On **Day 9**, we constructed the `HyperspectralPatchDataset` and PyTorch `DataLoader` pipelines for reproducible batching, spatial reflection padding, and zero-indexed class target mapping ($1..16 \to 0..15$).

Today (**Day 10**), we implement and execute the complete **supervised training and validation pipeline**:
1. Connect the real Salinas DataLoaders to the Day 8 3D-CNN.
2. Formulate the loss function (`nn.CrossEntropyLoss`) on raw logits.
3. Configure the AdamW optimizer with decoupled weight decay.
4. Execute an end-to-end smoke test to verify forward and backward propagation.
5. Train the model across multiple epochs, tracking training and validation loss and accuracy.
6. Save the best checkpoint based on validation loss using atomic writes.
7. Reload the checkpoint into a fresh model instance and verify forward inference.

### Scientific Task Framing: Multiclass Land-Cover Classification
The Salinas hyperspectral dataset represents an agricultural land-cover scene near Salinas Valley, California, captured by the NASA AVIRIS sensor. The objective is to classify each spatial pixel into one of 16 agricultural land-cover classes (e.g., Broccoli, Celery, Fallow, Grapes, Lettuce). This is a **multiclass land-cover classification benchmark**, not crop-disease diagnosis.

In [1]:
import sys
import time
from pathlib import Path

# Resolve project root dynamically whether notebook is run from root or notebooks/
current_dir = Path.cwd().resolve()
if (current_dir / "data" / "raw" / "Salinas_corrected.mat").is_file():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data" / "raw" / "Salinas_corrected.mat").is_file():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from terraspectra.data import (
    SalinasLoader,
    SALINAS_CLASS_NAMES,
    create_dataloaders,
)
from terraspectra.models import (
    Hyperspectral3DCNN,
    Hyperspectral3DCNNConfig,
    TrainingConfig,
    train_one_epoch,
    validate_one_epoch,
    train_model,
    save_checkpoint,
    load_checkpoint,
    select_device,
    set_seed,
)

print(f"Project Root:    {PROJECT_ROOT}")
print(f"PyTorch Version: {torch.__version__}")
device = select_device("auto")
print(f"Selected Device: {device}")
set_seed(42)

Project Root:    C:\Users\abhis\Desktop\Tetraspectra
PyTorch Version: 2.14.1+cpu
Selected Device: cpu


## 2. Dataset and Label Verification
We load the real Salinas dataset and inspect the ground-truth class distribution.
- Salinas scene dimensions: $512 \times 217$ pixels with 204 contiguous spectral bands.
- Ground truth contains 16 agricultural classes ($1..16$) and background ($0$).
- When `exclude_unlabeled=True`, background pixels are excluded, leaving **54,129 active labeled samples**.

In [2]:
# Ingest Salinas dataset using Day 2 loader
raw_data_dir = PROJECT_ROOT / "data" / "raw"
loader = SalinasLoader(raw_data_dir=raw_data_dir)
dataset = loader.load(allow_synthetic_fallback=True)

raw_cube = dataset.cube
gt = dataset.gt

print(f"Raw Cube Shape:         {raw_cube.shape} (H=512, W=217, Bands=204)")
print(f"Ground Truth Shape:     {gt.shape}")
print(f"Total Pixels:           {gt.size:,}")
print(f"Labeled Pixels (C1-C16):{int(np.sum(gt > 0)):,} (excluded unlabeled class 0)")

# Also load Day 4 PCA-reduced cube (15 principal components explaining >99.7% spectral variance)
pca_data_path = PROJECT_ROOT / "data" / "processed" / "salinas_pca_reduced.npz"
if pca_data_path.is_file():
    pca_npz = np.load(pca_data_path)
    pca_cube = pca_npz["reduced_cube"]
    print(f"PCA-Reduced Cube Shape: {pca_cube.shape} (15 principal components)")
else:
    pca_cube = raw_cube[:, :, :15]

Raw Cube Shape:         (512, 217, 204) (H=512, W=217, Bands=204)
Ground Truth Shape:     (512, 217)
Total Pixels:           111,104
Labeled Pixels (C1-C16):54,129 (excluded unlabeled class 0)
PCA-Reduced Cube Shape: (512, 217, 15) (15 principal components)


## 3. Model Architecture and Batch Layout
Our `Hyperspectral3DCNN` uses 3D convolutions `(Depth, Height, Width)` followed by `nn.AdaptiveAvgPool3d((1, 1, 1))` and a classification head.
- Input Tensor: `(Batch, Channels=1, Depth=D, Height=7, Width=7)`
- Output Tensor: `(Batch, num_classes=16)` (raw unnormalized logits)
- Loss: `nn.CrossEntropyLoss` combines `nn.LogSoftmax` and `nn.NLLLoss` into a single numerically stable operation.

In [3]:
# Construct DataLoaders on the 15-band PCA cube (or raw cube)
patch_size = 7
batch_size = 128

train_loader, val_loader, test_loader, meta = create_dataloaders(
    cube_or_dataset=pca_cube,
    gt=gt,
    patch_size=patch_size,
    batch_size=batch_size,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
    stratify=True,
    exclude_unlabeled=True,
)

print(f"Splits: Train={meta.train_samples:,}, Val={meta.val_samples:,}, Test={meta.test_samples:,}")
print(f"Classes: {meta.num_classes} (mapped {meta.label_to_idx[1]}..{meta.label_to_idx[16]} for C1..C16)")

# Inspect first batch
batch_x, batch_y = next(iter(train_loader))
print(f"\nBatch X Shape: {batch_x.shape} (Batch={batch_size}, Channels=1, Depth={pca_cube.shape[2]}, H={patch_size}, W={patch_size})")
print(f"Batch Y Shape: {batch_y.shape} (Labels in [{batch_y.min().item()}, {batch_y.max().item()}])")

# Instantiate model
model_config = Hyperspectral3DCNNConfig(
    in_channels=1,
    spectral_depth=pca_cube.shape[2],
    spatial_size=patch_size,
    num_classes=meta.num_classes,
    conv_channels=(8, 16, 32),
    kernel_spectral=(7, 5, 3),
    dropout_rate=0.3,
)
model = Hyperspectral3DCNN(model_config).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nHyperspectral3DCNN Parameter Count: {total_params:,} trainable parameters")

Splits: Train=37,891, Val=8,118, Test=8,120
Classes: 16 (mapped 0..15 for C1..C16)

Batch X Shape: torch.Size([128, 1, 15, 7, 7]) (Batch=128, Channels=1, Depth=15, H=7, W=7)
Batch Y Shape: torch.Size([128]) (Labels in [0, 15])

Hyperspectral3DCNN Parameter Count: 23,352 trainable parameters


## 4. Short Smoke Test: Forward, Loss, Backward, and Optimizer Step
Before executing full training, we run a minimal sanity check on a single mini-batch:
1. Move batch to device.
2. Reset gradients (`optimizer.zero_grad()`).
3. Compute forward pass (`logits = model(batch_x)`).
4. Evaluate cross-entropy loss.
5. Backpropagate gradients (`loss.backward()`).
6. Update model weights (`optimizer.step()`).

In [4]:
smoke_model = Hyperspectral3DCNN(model_config).to(device)
optimizer = torch.optim.AdamW(smoke_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# Single-step smoke test
smoke_bx = batch_x.to(device)
smoke_by = batch_y.to(device)

initial_param = next(smoke_model.parameters()).clone()

optimizer.zero_grad()
smoke_logits = smoke_model(smoke_bx)
smoke_loss = criterion(smoke_logits, smoke_by)
smoke_loss.backward()
optimizer.step()

updated_param = next(smoke_model.parameters())

print(f"Smoke Test Forward Pass: Logits Shape={smoke_logits.shape}")
print(f"Initial Cross-Entropy Loss: {smoke_loss.item():.4f}")
print(f"Parameter Update Verified:  {not torch.allclose(initial_param, updated_param)}")
assert torch.isfinite(smoke_loss)
print("[SUCCESS] Smoke test passed cleanly!")

Smoke Test Forward Pass: Logits Shape=torch.Size([128, 16])
Initial Cross-Entropy Loss: 2.8171
Parameter Update Verified:  True
[SUCCESS] Smoke test passed cleanly!


## 5. Model Training Execution
We execute a complete training run using our production training loop `train_model`:
- **Epochs**: 3
- **Batch Size**: 128
- **Learning Rate**: $10^{-3}$ with AdamW optimizer
- **Criterion**: `nn.CrossEntropyLoss`
- **Checkpointing**: Saved best model based on validation loss to `outputs/checkpoints/best_3d_cnn.pt`.

In [5]:
checkpoint_dir = PROJECT_ROOT / "outputs" / "checkpoints"
train_config = TrainingConfig(
    epochs=3,
    lr=1e-3,
    weight_decay=1e-4,
    device=str(device),
    checkpoint_dir=str(checkpoint_dir),
    checkpoint_filename="best_3d_cnn.pt",
    seed=42,
    monitor_metric="val_loss",
)

t0 = time.time()
trained_model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_config,
    metadata=meta,
    model_config=model_config,
)
total_train_time = time.time() - t0
print(f"\nTraining completed in {total_train_time:.1f} seconds.")
print(f"Best Validation Epoch: {history.best_epoch}")
print(f"Best Validation Loss:  {history.best_val_loss:.4f}")
print(f"Best Validation Acc:   {history.best_val_acc*100:.2f}%")


Training completed in 67.2 seconds.
Best Validation Epoch: 1
Best Validation Loss:  0.0903
Best Validation Acc:   97.11%


## 6. Training and Validation Curves
Visualizing loss descent and accuracy curves across epochs provides critical diagnostics into optimization dynamics and generalization.

In [6]:
epochs_range = [ep.epoch for ep in history.epochs]
train_losses = [ep.train_loss for ep in history.epochs]
val_losses = [ep.val_loss for ep in history.epochs]
train_accs = [ep.train_acc * 100 for ep in history.epochs]
val_accs = [ep.val_acc * 100 for ep in history.epochs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Loss plot
ax1.plot(epochs_range, train_losses, marker="o", color="#1f77b4", lw=2, label="Train Loss")
ax1.plot(epochs_range, val_losses, marker="s", color="#ff7f0e", lw=2, linestyle="--", label="Val Loss")
ax1.axvline(history.best_epoch, color="#2ca02c", linestyle=":", label=f"Best Checkpoint (Epoch {history.best_epoch})")
ax1.set_title("Cross-Entropy Loss vs. Epoch", fontsize=11, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Mean Cross-Entropy Loss")
ax1.set_xticks(epochs_range)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend()

# Accuracy plot
ax2.plot(epochs_range, train_accs, marker="o", color="#1f77b4", lw=2, label="Train Accuracy")
ax2.plot(epochs_range, val_accs, marker="s", color="#ff7f0e", lw=2, linestyle="--", label="Val Accuracy")
ax2.axvline(history.best_epoch, color="#2ca02c", linestyle=":", label=f"Best Checkpoint (Epoch {history.best_epoch})")
ax2.set_title("Classification Accuracy vs. Epoch", fontsize=11, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_xticks(epochs_range)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

C:\Users\abhis\AppData\Local\Temp\ipykernel_25656\3583523719.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Checkpoint Reload and Verification
We verify that the saved checkpoint file can be loaded into an entirely new, uninitialized `Hyperspectral3DCNN` instance and execute inference accurately.

In [7]:
checkpoint_path = Path(history.checkpoint_path)
print(f"Checkpoint File: {checkpoint_path}")
print(f"File Size:       {checkpoint_path.stat().st_size / (1024*1024):.2f} MB")

# Instantiate fresh uninitialized model
fresh_model = Hyperspectral3DCNN(model_config)
payload = load_checkpoint(checkpoint_path, model=fresh_model, device=device)

print(f"Loaded Checkpoint Metadata:")
print(f"  Epoch:        {payload['epoch']}")
print(f"  Val Loss:     {payload['val_loss']:.4f}")
print(f"  Val Accuracy: {payload['val_acc']*100:.2f}%")
print(f"  Num Classes:  {payload['num_classes']}")

# Verify forward pass on loaded model
fresh_model.eval()
with torch.no_grad():
    sample_logits = fresh_model(batch_x.to(device))
    preds = sample_logits.argmax(dim=-1).cpu()

print(f"\nInference Verification on Fresh Model:")
print(f"  Output Logits Shape: {sample_logits.shape}")
print(f"  First 10 Predictions: {preds[:10].tolist()}")
print(f"  First 10 Targets:     {batch_y[:10].tolist()}")
acc_batch = (preds == batch_y).float().mean().item() * 100
print(f"  Batch Sample Accuracy: {acc_batch:.1f}%")
print("[SUCCESS] Checkpoint successfully verified!")

Checkpoint File: C:\Users\abhis\Desktop\Tetraspectra\outputs\checkpoints\best_3d_cnn.pt
File Size:       0.29 MB
Loaded Checkpoint Metadata:
  Epoch:        1
  Val Loss:     0.0903
  Val Accuracy: 97.11%
  Num Classes:  16

Inference Verification on Fresh Model:
  Output Logits Shape: torch.Size([128, 16])
  First 10 Predictions: [14, 6, 6, 7, 14, 7, 8, 7, 14, 6]
  First 10 Targets:     [7, 6, 6, 7, 14, 14, 8, 7, 7, 6]
  Batch Sample Accuracy: 95.3%
[SUCCESS] Checkpoint successfully verified!


## 8. What I Learned & Interview Preparation

| Concept | Explanation |
|---|---|
| **Forward Pass** | Input tensor $(B, 1, D, H, W)$ passes through Conv3D blocks, BatchNorm, ReLU, Adaptive 3D Pooling, and linear projection to output unnormalized class logits $(B, 16)$. |
| **Loss Calculation** | `nn.CrossEntropyLoss` compares predicted logits with ground-truth class indices. It applies log-softmax internally for numerical stability. |
| **Backpropagation** | Computes analytical gradients $\frac{\partial \mathcal{L}}{\partial \mathbf{W}}$ of the loss with respect to all trainable parameters via the chain rule. |
| **Optimizer Step** | AdamW updates weight tensors $\mathbf{W} \leftarrow \mathbf{W} - \eta \cdot \frac{\hat{\mathbf{m}}}{\sqrt{\hat{\mathbf{v}}} + \epsilon} - \lambda \mathbf{W}$, using adaptive first and second moment estimates with decoupled weight decay. |
| **Epoch vs. Batch** | A batch is a single mini-partition of samples ($N=128$) evaluated in one gradient step. An epoch represents one complete pass over the entire training dataset ($N=37,891$). |
| **Train vs. Validation** | Training data updates parameters via gradients. Validation data evaluates generalization without gradients and determines early stopping / checkpoint selection. |
| **Why Select Best on Val?** | Training loss decreases monotonically as weights fit data, potentially overfitting. The lowest validation loss identifies the point of optimal generalization. |

---

### What's Next (Day 11):
On **Day 11**, we will evaluate this trained checkpoint thoroughly on the held-out **Test Set** (8,120 samples), generating per-class precision, recall, F1-scores, overall accuracy (OA), average accuracy (AA), Kappa coefficient ($\kappa$), and a full confusion matrix.